# Analyse the Results of Running Moran Process Experiment on Different Graphs
This is the newest version of this analysis file, where I can merge the csv of different jobs. 

## Setup

Imports, configuration, and data loading. Set `BATCH_NAME` below to point at whichever batch you want to analyze.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process 

import sys
sys.path.insert(0, 'src')

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from moran_process.analysis.analysis_utils import *

# change this if on a different computer!
from moran_process.core.population_graph import GRAPH_PROPS
# Set aesthetic parameters for "publication-quality" plots
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.2)
plt.rcParams['figure.figsize'] = (5.5, 4.5)
plt.rcParams['lines.linewidth'] = 2.5


In [ ]:

BATCH_NAME = '2026-06-10_test-everything-still-works-3'  # <-- change this to whatever batch you want to analyse
CATEGORY_FILTER = None  # set to e.g. 'Random', 'Complete', 'Cycle' to filter; None = all categories
ANIMAL_CATEGORIES = ["Mammalian", "Fish", "Avian"]



In [ ]:
# --- Batch provenance ---
ROOT = Path(os.getcwd()) 

# Now define your paths relative to ROOT
BATCH_DIR = ROOT / "simulation_data" / BATCH_NAME
FIGURES_DIR = BATCH_DIR / "figures"

# To register a new batch, run create_batch_info(BATCH_DIR, name=..., description=..., ...)
batch_info = load_batch_info(BATCH_DIR)
BATCH_LABEL = batch_info['name']
plot_batch_info_card(batch_info, figures_dir=FIGURES_DIR)

In [ ]:
import glob

tmp_results_path = os.path.join(BATCH_DIR, "tmp", "results")
parquet_files = glob.glob(os.path.join(tmp_results_path, "raw_results_job_*.parquet"))
csv_files = glob.glob(os.path.join(tmp_results_path, "raw_results_job_*.csv"))
all_files = parquet_files or csv_files
fmt = "parquet" if parquet_files else "csv"
print(f"Found {len(all_files)} {fmt} files in temp results directory: {tmp_results_path}.")


### The Graph Features we analyse: 


In [ ]:
GRAPH_PROPERTY_DESCRIPTION
for prop, desc in GRAPH_PROPERTY_DESCRIPTION.items():
    print(f"{prop.title().replace('_', ' '):40}: {desc}")

In [ ]:
results_df_path = aggregate_results_no_load(batch_dir=BATCH_DIR, delete_temp=False)

In [ ]:
_rp = Path(str(results_df_path))
if _rp.suffix == '.parquet':
    lazy_df = pl.scan_parquet(str(results_df_path))
else:
    lazy_df = pl.scan_csv(str(results_df_path))
schema = lazy_df.collect_schema()
n_rows = lazy_df.select(pl.len()).collect()[0, 0]
print("columns:", list(schema.keys()))
print(f"Raw data: {n_rows:,} rows x {len(schema)} columns")


In [ ]:
from moran_process.analysis.batch_speed_report import batch_speed_report
df_speed = lazy_df.select(["job_id", "steps", "duration"]).collect().to_pandas()
batch_speed_report(BATCH_NAME, df_speed, BATCH_DIR)

In [ ]:
# --- Aggregating the Raw Simulations Result ---

df_graphs = pd.read_csv(os.path.join(BATCH_DIR, 'graph_props.csv'))
graph_statistics_path = BATCH_DIR / "graph_statistics.csv"

analysis_df = build_graph_statistics(
    results_df_path, df_graphs, graph_statistics_path, category_filter=CATEGORY_FILTER
)
analysis_df.tail(20)


In [ ]:
plot_steps_histogram(
    analysis_df,
    metric='mean_steps',
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
plot_steps_histogram(
    analysis_df,
    metric='mean_steps',
    category='Random',
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

In [ ]:
categories = sorted(analysis_df['category'].dropna().unique().tolist())
category_color_dict = generate_robust_color_dict(analysis_df, CATEGORY_COLOR_DICT)

print("Color dictionary:")
for cat, color in category_color_dict.items():
    print(f"  {cat:15}: {color}")

In [ ]:
plot_steps_violin(
    results_path=results_df_path,
    df_graphs=df_graphs,
    color_dict=category_color_dict,
    categories=categories,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)


In [ ]:
plot_outcome_vs_property(
    analysis_df,
    'std_steps',
    'mean_steps',
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)   

In [ ]:
analysis_df_single_r = analysis_df[analysis_df['r'] == 1.1]

plot_outcome_vs_property(
    analysis_df,
    'mean_steps',
    'prob_fixation',
    density_threshold=None,
    color_dict=category_color_dict,
    highlight_categories=ANIMAL_CATEGORIES,
    figures_dir=FIGURES_DIR,
    batch_name=BATCH_LABEL,
)

## Per-Property Effect on Fixation Time (`r = 1.1`)

Each plot below sweeps one graph structural property against mean fixation time under weak selection (`r = 1.1`). Animal graphs are highlighted; random graphs form the background distribution.  Look for properties where the animal graphs sit outside the random cloud -- those are structural features that may explain amplifier or suppressor behavior.

In [ ]:
for prop in GRAPH_PROPS:
    plot_outcome_vs_property(
        analysis_df_single_r, prop, 'mean_steps',
        density_threshold=50,
        color_dict=category_color_dict,
        highlight_categories=ANIMAL_CATEGORIES,
        figures_dir=FIGURES_DIR,
        batch_name=BATCH_LABEL,
    )

## Per-Property Effect on Fixation Probability (`r = 1.1`)

Same sweep as above but with fixation probability on the y-axis. Cross-reference with the fixation time plots: a true amplifier should show both elevated probability *and* shorter fixation time relative to the neutral drift baseline (`rho = 1/N`).

In [ ]:
plot_two_property_effect(analysis_df_single_r, 
                                'density', 
                                'n_nodes', 
                                'mean_steps',
                                color_dict=category_color_dict,
                                )

In [ ]:
plot_two_property_effect(analysis_df_single_r, 
                                'n_edges', 
                                'n_nodes', 
                                'mean_steps',
                                color_dict=category_color_dict,
                                )

In [ ]:
for prop in GRAPH_PROPS:
    plot_outcome_vs_property(
        analysis_df_single_r, prop, 'prob_fixation',
        density_threshold=50,
        color_dict=category_color_dict,
        highlight_categories=ANIMAL_CATEGORIES,
        figures_dir=FIGURES_DIR,
        batch_name=BATCH_LABEL,
    )

In [ ]:
plot_outcome_vs_property(
    analysis_df_single_r,
    'degree_std',
    'mean_steps',
    density_threshold=50,
    color_dict=category_color_dict,
    batch_name=BATCH_LABEL,
)

plt.figure(figsize=(10, 8))
plt.hexbin(analysis_df_single_r['degree_std'], analysis_df_single_r['mean_steps'], gridsize=20, cmap='YlOrRd', mincnt=1)
plt.xlabel('degree_std')
plt.ylabel('mean_steps')
plt.colorbar(label='count')
plt.title('Hexbin plot: degree_std vs mean_steps')
plt.show()